# Glyph Agent

**Run the following code first to save the model**

In [ ]:
import torch
import torch.nn as nn
import src.machine_learning as ML
import os
from src.Model_BR import GlyphClassifier
import matplotlib.pyplot as plt
import pandas as pd
import torch.nn.functional as F
import numpy as np
from torch.optim.lr_scheduler import StepLR
from datetime import datetime


config={
    "architecture": "CNN-Glyph",
    "dataset": 'data/simple-star.zip',
    "test": 'data/simple-star-validation.zip',
    "epochs": 10,
    "batch_size": 64,
    "learning_rate": 0.0005,
    "loss_fn": "MSELoss",
    "optimizer": "Adam",
    "image_resolution": (128, 128),
    "regression": True,
    "num_bins": 5,
    "rotation": 0,
    "translation": 0
}

# Getting the dataset
dataset_file = config["dataset"]
test_file = config["test"]
train_dataset = ML.GlyphDataset(dataset_file, resize=config["image_resolution"], split = "train",augmentation_rot=config["rotation"],augmentation_tran=config["translation"])
validation_dataset = ML.GlyphDataset(dataset_file, resize=config["image_resolution"],split = 'test',augmentation_rot=config["rotation"],augmentation_tran=config["translation"])
test_dataset = ML.GlyphDataset(test_file, resize=config["image_resolution"], split='test',augmentation_rot=config["rotation"],augmentation_tran=config["translation"])

# Assign the loaders 

train_loader = ML.create_loader(train_dataset, batch_size=config["batch_size"], shuffle = True)
test_loader = ML.create_loader(test_dataset, batch_size=config["batch_size"], shuffle = False)
validation_loader = ML.create_loader(validation_dataset, batch_size=config["batch_size"], shuffle=False)



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

bin_centers = torch.linspace(0, 100, config["num_bins"] + 1, device=device)[:-1] + 50 / config["num_bins"]

model = GlyphClassifier(resolution=config["image_resolution"], NUM_bins=config["num_bins"]).to(device)

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])

scheduler = StepLR(optimizer, step_size=5, gamma=0.5)  # Reduce LR by half every 5 epochs

experiment_name = f"exp-SimpleStar-{config['image_resolution'][0]}x{config['image_resolution'][1]}-{config['num_bins']}bins-BinnedRegression-withvalidation"

print(f"Experiment name: {experiment_name}")

# Initialize W&B

train_losses = []
epoch_train_losses = []
val_losses = []
global_step = 0

for epoch in range(config["epochs"]):
    model.train()
    running_loss = 0.0

    for images, values in train_loader:
        images = images.to(device)
        values = values.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        probabilities = torch.softmax(outputs, dim=1)
        predictions = torch.sum(probabilities * bin_centers, dim=1)
        loss = criterion(predictions, values)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        train_losses.append(loss.item())

        mae = F.l1_loss(predictions, values).item()

        if global_step % 100 == 0:
            print(f"Step {global_step}: Loss = {loss.item():.4f}")
        global_step += 1

    avg_train_loss = running_loss / len(train_loader)
    epoch_train_losses.append(avg_train_loss)
    print(f"Epoch {epoch+1}/{config['epochs']} - Train Loss: {avg_train_loss:.4f}")

    # Validation at the end of the epoch 
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, values in validation_loader:
            images = images.to(device)
            values = values.to(device)
            logits = model(images)
            probabilities = F.softmax(logits, dim=1)
            preds = torch.sum(probabilities * bin_centers, dim=1)
            val_loss += criterion(preds, values).item()
    
    scheduler.step()

    avg_val_loss = val_loss / len(validation_loader)
    val_losses.append(avg_val_loss)
    print(f"Epoch {epoch+1}/{config['epochs']} - Val Loss: {avg_val_loss:.4f}")


# === Save the model locally ===
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_dir = "saved_models"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, f"GlyphAgent_{timestamp}.pt")
torch.save(model.state_dict(), "model.pt")

print(f"\n✅ Model saved to: {model_path}")


**GlyphAgent showcase**

In [ ]:
from src.glyph_agent import GlyphAgent

agent = GlyphAgent("data/simple-star-validation.zip", "model.pt", name="StarAI", device="cuda:0")

task = {"x1": 42.9, "x2": 81.3, "distance": 100}
response = agent.get_response(task)

print(response)